# Step 1: Aggregate Prior Commitments

Goal: Create one row per cdcno with all prior history features

## Load Prior Commitments Data

Loads the raw prior commitments file from GitHub. Each row represents one prior offense for one person. The same person (cdcno) can appear multiple times.

In [1]:
import pandas as pd

prior_url = "https://raw.githubusercontent.com/redoio/offenses_data/main/data/prior_commitments.csv"
prior = pd.read_csv(prior_url)

print(f"Total rows: {len(prior):,}")
print(f"Unique people: {prior['cdcno'].nunique():,}")

Total rows: 191,436
Unique people: 43,498


## Count Offenses by Category per Person

Groups by cdcno and offense category, counts how many offenses of each type each person has. Pivots so each category becomes a column. Adds a total count column.

In [2]:
category_counts = prior.groupby(['cdcno', 'offense category']).size().unstack(fill_value=0)
category_counts.columns = [f"count_prior_{col.lower().replace(' ', '_')}" for col in category_counts.columns]
category_counts['count_prior_total'] = category_counts.sum(axis=1)
category_counts = category_counts.reset_index()

print(category_counts.head(10))
print(f"\nShape: {category_counts.shape}")

        cdcno  count_prior_case_enhancement  \
0  00009164d5                             0   
1  0000a5860b                             0   
2  0003140564                             0   
3  000314c92e                             0   
4  0006e25501                             3   
5  0007893255                             0   
6  000ad3cb30                             0   
7  001689b264                             0   
8  00173d8423                             1   
9  00185ed6f6                             0   

   count_prior_crimes_against_persons  count_prior_drug_crimes  \
0                                   0                        1   
1                                   1                        1   
2                                   2                        0   
3                                   1                        0   
4                                   2                        0   
5                                   0                        0   
6                   

## Count Prior Enhancements

Counts how many prior offenses were classified as 'Case Enhancement' for each person.

In [3]:
prior['has_enhancement'] = (prior['offense category'] == 'Case Enhancement').astype(int)
enhancements = prior.groupby('cdcno')['has_enhancement'].sum().reset_index()
enhancements.columns = ['cdcno', 'count_prior_enhancements']

print(enhancements.head(10))
print(f"\nPeople with prior enhancements: {(enhancements['count_prior_enhancements'] > 0).sum():,}")

        cdcno  count_prior_enhancements
0  00009164d5                         0
1  0000a5860b                         0
2  0003140564                         0
3  000314c92e                         0
4  0006e25501                         3
5  0007893255                         0
6  000ad3cb30                         0
7  001689b264                         0
8  00173d8423                         1
9  00185ed6f6                         0

People with prior enhancements: 10,859


## Count In-Prison Prior Offenses

Counts how many prior offenses were committed while the person was already in prison (in prison = 'In-Prison').

In [4]:
prior['in_prison_flag'] = (prior['in prison'] == 'In-Prison').astype(int)
in_prison = prior.groupby('cdcno')['in_prison_flag'].sum().reset_index()
in_prison.columns = ['cdcno', 'count_prior_in_prison']

print(in_prison.head(10))
print(f"\nPeople with in-prison priors: {(in_prison['count_prior_in_prison'] > 0).sum():,}")

        cdcno  count_prior_in_prison
0  00009164d5                      0
1  0000a5860b                      0
2  0003140564                      0
3  000314c92e                      0
4  0006e25501                      0
5  0007893255                      0
6  000ad3cb30                      0
7  001689b264                      0
8  00173d8423                      0
9  00185ed6f6                      0

People with in-prison priors: 2,442


## Merge All Prior Features

Combines offense counts, enhancement counts, and in-prison counts into one dataframe with one row per person.

In [5]:
prior_features = category_counts.merge(enhancements, on='cdcno', how='left')
prior_features = prior_features.merge(in_prison, on='cdcno', how='left')

prior_features['count_prior_enhancements'] = prior_features['count_prior_enhancements'].fillna(0).astype(int)
prior_features['count_prior_in_prison'] = prior_features['count_prior_in_prison'].fillna(0).astype(int)

print(prior_features.head(10))
print(f"\nShape: {prior_features.shape}")

        cdcno  count_prior_case_enhancement  \
0  00009164d5                             0   
1  0000a5860b                             0   
2  0003140564                             0   
3  000314c92e                             0   
4  0006e25501                             3   
5  0007893255                             0   
6  000ad3cb30                             0   
7  001689b264                             0   
8  00173d8423                             1   
9  00185ed6f6                             0   

   count_prior_crimes_against_persons  count_prior_drug_crimes  \
0                                   0                        1   
1                                   1                        1   
2                                   2                        0   
3                                   1                        0   
4                                   2                        0   
5                                   0                        0   
6                   

## Verify One Row Per Person

Confirms that the final dataset has exactly one row per cdcno with no duplicates.

In [6]:
print(f"Unique cdcno: {prior_features['cdcno'].nunique():,}")
print(f"Total rows: {len(prior_features):,}")
print(f"Match: {prior_features['cdcno'].nunique() == len(prior_features)}")

Unique cdcno: 43,498
Total rows: 43,498
Match: True


# Step 2: Aggregate Current Commitments

Goal: Create one row per cdcno with all current offense features

## Load Current Commitments Data

Loads the raw current commitments file from GitHub. Each row represents one current offense for one person. The same person (cdcno) can appear multiple times.

In [7]:
current_url = "https://raw.githubusercontent.com/redoio/offenses_data/main/data/current_commitments.csv"
current = pd.read_csv(current_url)

print(f"Total rows: {len(current):,}")
print(f"Unique people: {current['cdcno'].nunique():,}")

Total rows: 369,125
Unique people: 95,476


C:\Users\vgwin\AppData\Local\Temp\ipykernel_46788\3134961440.py:2: DtypeWarning: Columns (7,22,23,24,25,26,27,28,29,30,31,32,33) have mixed types. Specify dtype option on import or set low_memory=False.
  current = pd.read_csv(current_url)


## Count Offenses by Category per Person

Groups by cdcno and offense category, counts how many current offenses of each type each person has. Pivots so each category becomes a column. Adds a total count column.

In [8]:
category_counts = current.groupby(['cdcno', 'offense category']).size().unstack(fill_value=0)
category_counts.columns = [f"count_current_{col.lower().replace(' ', '_')}" for col in category_counts.columns]
category_counts['count_current_total'] = category_counts.sum(axis=1)
category_counts = category_counts.reset_index()

print(category_counts.head(10))
print(f"\nShape: {category_counts.shape}")

        cdcno  count_current_case_enhancement  \
0  00009164d5                               0   
1  0000a5860b                               1   
2  00015ca40d                               0   
3  0002afe811                               0   
4  0003140564                               0   
5  000314c92e                               0   
6  0003236494                               0   
7  0003668176                               0   
8  0003b2f201                               0   
9  0004943069                               0   

   count_current_crimes_against_persons  count_current_drug_crimes  \
0                                     4                          0   
1                                     4                          0   
2                                     3                          0   
3                                     1                          0   
4                                     1                          0   
5                                     0 

## Count Current Enhancements

Counts how many enhancement columns (off enh1 through off enh11) have values for each person.

In [9]:
enh_cols = [f'off enh{i}' for i in range(1, 12)]
current['count_enhancements'] = current[enh_cols].notna().sum(axis=1)
enhancements = current.groupby('cdcno')['count_enhancements'].sum().reset_index()
enhancements.columns = ['cdcno', 'count_current_enhancements']

print(enhancements.head(10))
print(f"\nPeople with current enhancements: {(enhancements['count_current_enhancements'] > 0).sum():,}")

        cdcno  count_current_enhancements
0  00009164d5                           1
1  0000a5860b                           2
2  00015ca40d                           0
3  0002afe811                           1
4  0003140564                           0
5  000314c92e                           0
6  0003236494                           2
7  0003668176                           2
8  0003b2f201                           0
9  0004943069                           5

People with current enhancements: 45,075


## Count In-Prison Current Offenses

Counts how many current offenses were committed while the person was already in prison (in prison = 'In-Prison').

In [10]:
current['in_prison_flag'] = (current['in prison'] == 'In-Prison').astype(int)
in_prison = current.groupby('cdcno')['in_prison_flag'].sum().reset_index()
in_prison.columns = ['cdcno', 'count_current_in_prison']

print(in_prison.head(10))
print(f"\nPeople with in-prison current offenses: {(in_prison['count_current_in_prison'] > 0).sum():,}")

        cdcno  count_current_in_prison
0  00009164d5                        0
1  0000a5860b                        2
2  00015ca40d                        0
3  0002afe811                        0
4  0003140564                        0
5  000314c92e                        0
6  0003236494                        0
7  0003668176                        0
8  0003b2f201                        0
9  0004943069                        0

People with in-prison current offenses: 10,475


## Merge All Current Features

Combines offense counts, enhancement counts, and in-prison counts into one dataframe with one row per person.

In [11]:
current_features = category_counts.merge(enhancements, on='cdcno', how='left')
current_features = current_features.merge(in_prison, on='cdcno', how='left')

current_features['count_current_enhancements'] = current_features['count_current_enhancements'].fillna(0).astype(int)
current_features['count_current_in_prison'] = current_features['count_current_in_prison'].fillna(0).astype(int)

print(current_features.head(10))
print(f"\nShape: {current_features.shape}")

        cdcno  count_current_case_enhancement  \
0  00009164d5                               0   
1  0000a5860b                               1   
2  00015ca40d                               0   
3  0002afe811                               0   
4  0003140564                               0   
5  000314c92e                               0   
6  0003236494                               0   
7  0003668176                               0   
8  0003b2f201                               0   
9  0004943069                               0   

   count_current_crimes_against_persons  count_current_drug_crimes  \
0                                     4                          0   
1                                     4                          0   
2                                     3                          0   
3                                     1                          0   
4                                     1                          0   
5                                     0 

## Verify One Row Per Person

Confirms that the final dataset has exactly one row per cdcno with no duplicates.

In [12]:
print(f"Unique cdcno: {current_features['cdcno'].nunique():,}")
print(f"Total rows: {len(current_features):,}")
print(f"Match: {current_features['cdcno'].nunique() == len(current_features)}")

Unique cdcno: 95,476
Total rows: 95,476
Match: True


# Step 3: Create Final Analysis Dataset

Goal: Combine demographics (base) with prior features and current features into one master dataset

## Load Demographics Data

Loads demographics which is already one row per person. This serves as the base dataset.

In [13]:
demographics_url = "https://raw.githubusercontent.com/redoio/offenses_data/main/data/demographics.csv"
demographics = pd.read_csv(demographics_url)

print(f"Total rows: {len(demographics):,}")
print(f"Unique people: {demographics['cdcno'].nunique():,}")

Total rows: 95,476
Unique people: 95,476


## Select Key Demographics Columns

Keeps only the columns needed for analysis: cdcno, ethnicity, sentence info, and county. Offense category is excluded since it's already captured in current offense counts.

In [14]:
demographics_clean = demographics[[
    'cdcno',
    'ethnicity',
    'controlling case sentencing county',
    'sentence type',
    'aggregate sentence in months'
]].copy()

print(demographics_clean.head(10))
print(f"\nShape: {demographics_clean.shape}")

        cdcno ethnicity controlling case sentencing county     sentence type  \
0  2cf2a233c4     Black                     San Bernardino    Second Striker   
1  5a72696541     White                         Sacramento  Life with Parole   
2  7d608b6a4c     White                              Butte  Life with Parole   
3  39c1bc8c2f     Other                              Butte     Third Striker   
4  220f2cdfc5     Black                        Los Angeles  Life with Parole   
5  82c8099e11     White                             Fresno  Life with Parole   
6  a97c5041d6     White                          San Diego  Life with Parole   
7  57d0dbcd13     White                        Los Angeles  Life with Parole   
8  221b46cbec     White                        Los Angeles  Life with Parole   
9  4151566ebc     Black                          Riverside    Second Striker   

   aggregate sentence in months  
0                            32  
1                           360  
2                

## Cap Extreme Sentence Values

Caps aggregate sentence at 1080 months (90 years) to prevent extreme outliers from skewing the analysis. Values above this are replaced with 1080.

In [15]:
demographics_clean['aggregate sentence in months'] = demographics_clean['aggregate sentence in months'].clip(upper=1080)

print(f"Max sentence after capping: {demographics_clean['aggregate sentence in months'].max()}")
print(f"Mean sentence: {demographics_clean['aggregate sentence in months'].mean():.2f}")

Max sentence after capping: 1080
Mean sentence: 332.97


## Merge with Prior Features

Left join demographics with prior_features. People with no prior history will have NaN values which we'll fill with 0.

In [16]:
analysis_data = demographics_clean.merge(prior_features, on='cdcno', how='left')

prior_cols = [c for c in analysis_data.columns if c.startswith('count_prior')]
analysis_data[prior_cols] = analysis_data[prior_cols].fillna(0).astype(int)

print(analysis_data.head(10))
print(f"\nShape: {analysis_data.shape}")

        cdcno ethnicity controlling case sentencing county     sentence type  \
0  2cf2a233c4     Black                     San Bernardino    Second Striker   
1  5a72696541     White                         Sacramento  Life with Parole   
2  7d608b6a4c     White                              Butte  Life with Parole   
3  39c1bc8c2f     Other                              Butte     Third Striker   
4  220f2cdfc5     Black                        Los Angeles  Life with Parole   
5  82c8099e11     White                             Fresno  Life with Parole   
6  a97c5041d6     White                          San Diego  Life with Parole   
7  57d0dbcd13     White                        Los Angeles  Life with Parole   
8  221b46cbec     White                        Los Angeles  Life with Parole   
9  4151566ebc     Black                          Riverside    Second Striker   

   aggregate sentence in months  count_prior_case_enhancement  \
0                            32                       

## Merge with Current Features

Left join with current_features. People with no matching current commitments will have NaN values which we'll fill with 0.

In [17]:
analysis_data = analysis_data.merge(current_features, on='cdcno', how='left')

current_cols = [c for c in analysis_data.columns if c.startswith('count_current')]
analysis_data[current_cols] = analysis_data[current_cols].fillna(0).astype(int)

print(analysis_data.head(10))
print(f"\nShape: {analysis_data.shape}")

        cdcno ethnicity controlling case sentencing county     sentence type  \
0  2cf2a233c4     Black                     San Bernardino    Second Striker   
1  5a72696541     White                         Sacramento  Life with Parole   
2  7d608b6a4c     White                              Butte  Life with Parole   
3  39c1bc8c2f     Other                              Butte     Third Striker   
4  220f2cdfc5     Black                        Los Angeles  Life with Parole   
5  82c8099e11     White                             Fresno  Life with Parole   
6  a97c5041d6     White                          San Diego  Life with Parole   
7  57d0dbcd13     White                        Los Angeles  Life with Parole   
8  221b46cbec     White                        Los Angeles  Life with Parole   
9  4151566ebc     Black                          Riverside    Second Striker   

   aggregate sentence in months  count_prior_case_enhancement  \
0                            32                       

## Verify Final Dataset

Confirms one row per person and shows final column list.

In [18]:
print(f"Unique cdcno: {analysis_data['cdcno'].nunique():,}")
print(f"Total rows: {len(analysis_data):,}")
print(f"Match: {analysis_data['cdcno'].nunique() == len(analysis_data)}")
print(f"\nTotal columns: {len(analysis_data.columns)}")
print(f"\nColumn list:\n{analysis_data.columns.tolist()}")

Unique cdcno: 95,476
Total rows: 95,476
Match: True

Total columns: 21

Column list:
['cdcno', 'ethnicity', 'controlling case sentencing county', 'sentence type', 'aggregate sentence in months', 'count_prior_case_enhancement', 'count_prior_crimes_against_persons', 'count_prior_drug_crimes', 'count_prior_other_crimes', 'count_prior_property_crimes', 'count_prior_total', 'count_prior_enhancements', 'count_prior_in_prison', 'count_current_case_enhancement', 'count_current_crimes_against_persons', 'count_current_drug_crimes', 'count_current_other_crimes', 'count_current_property_crimes', 'count_current_total', 'count_current_enhancements', 'count_current_in_prison']


## Save Final Analysis Dataset

Saves the complete analysis dataset to CSV for use in statistical models.

In [19]:
analysis_data.to_csv('analysis_data.csv', index=False)
print("Saved: analysis_data.csv")
print(f"Rows: {len(analysis_data):,}")
print(f"Columns: {len(analysis_data.columns)}")

Saved: analysis_data.csv
Rows: 95,476
Columns: 21
